# Big Data Processing Pipeline Using Apache Spark & PySpark
**Project**: Credit Card Fraud Detection Using Machine Learning and Big Data Analytics  
**Framework**: Apache Spark 4.2.0, Hadoop HDFS architecture  
**Objective**: Distributed data ingestion, hourly fraud velocity aggregation, and amount risk profiling.

---

## 1. Initialize Distributed SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, floor, count, sum as spark_sum, avg, round as spark_round, when

spark = SparkSession.builder \
    .appName('CreditCardFraudNotebook') \
    .master('local[*]') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession initialized. Version: {spark.version}')

## 2. Ingest Dataset into Distributed DataFrame

In [2]:
data_path = '../data/raw/creditcard.csv'
df_spark = spark.read.option('header', 'true').option('inferSchema', 'true').csv(data_path)
print(f'Total Ingested Records: {df_spark.count():,}')
df_spark.printSchema()

## 3. Distributed Transformations: Hourly Velocity and Risk Buckets

In [3]:
# Feature engineering in Spark
df_enhanced = df_spark.withColumn('Elapsed_Hour', floor(col('Time') / 3600)) \
                      .withColumn('Amount_Bracket',
                                  when(col('Amount') < 20.0, '1. Micro ($0-$20)')
                                  .when((col('Amount') >= 20.0) & (col('Amount') < 100.0), '2. Low ($20-$100)')
                                  .when((col('Amount') >= 100.0) & (col('Amount') < 500.0), '3. Medium ($100-$500)')
                                  .otherwise('4. High ($500+)'))

df_enhanced.select('Time', 'Elapsed_Hour', 'Amount', 'Amount_Bracket', 'Class').show(5)

## 4. Distributed Aggregations
Computing fraud rates across monetary brackets using Spark Catalyst Optimizer.

In [4]:
bracket_summary = df_enhanced.groupBy('Amount_Bracket').agg(
    count('Class').alias('Total_Transactions'),
    spark_sum('Class').alias('Fraud_Transactions'),
    spark_round(avg('Class') * 100, 4).alias('Fraud_Rate_Pct')
).orderBy('Amount_Bracket')

bracket_summary.show()

## 5. Stop Spark Session

In [5]:
spark.stop()
print('SparkSession successfully terminated.')